# DataPilot AI — Priorities 9–10: experiments (Colab T4)

**What:** Run held-out Dataset C against Systems A (`baseline_llm`) and B (`rag_only`), then B vs C (`finetuned_rag`). Rebuild result tables and figures from artefacts.

**Why:** The research question compares a generic LLM with domain RAG ± LoRA. Automatic metrics (point coverage, token F1, ROUGE-L, latency) are proxies, not human grades.

**Do not** use `--mock-llm` here. Mock runs validate the harness only.

**Already executed locally (no LLM):** retrieval top-k study `exp_05`. Training metrics come from `colab_t4_qlora_v1`.

Runtime → **GPU** → **T4**. After the run, copy `experiments/results/evaluation/exp_01_*` and `exp_02_*` back to the local repo and run `python scripts/build_results_tables.py`.

## 1. Setup

Point `PROJECT_DIR` at the uploaded project (must include `config/`, `data/evaluation/`, `knowledge_base/vector_store/`, `src/`, and the LoRA `adapter/` for System C).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# EDIT this path to your Masters_Project folder on Drive
PROJECT_DIR = "/content/drive/MyDrive/Masters_Project"

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
# %cd is required: os.chdir does not apply to !python shell cells
%cd /content/drive/MyDrive/Masters_Project
print("cwd:", os.getcwd())

## 2. Dependencies

Same Colab 2026 constraint as training: uninstall bitsandbytes/torchao. Qwen 1.5B inference uses fp16 on T4 (`HuggingFaceLLM` already falls back when 4-bit is unavailable).

In [ ]:
%pip uninstall -y bitsandbytes torchao
%pip -q install -U peft trl accelerate datasets pyyaml python-dotenv rouge-score matplotlib seaborn
%pip -q install -U faiss-cpu sentence-transformers

import torch
assert torch.cuda.is_available(), "Enable GPU runtime (T4) before LLM experiments"
print("torch", torch.__version__)
print(torch.cuda.get_device_name(0))

## 3. Optional smoke (5 questions)

Confirms Systems A/B load and write `summary.json`. **Not** thesis scores — too small and not the full set.

In [ ]:
%cd /content/drive/MyDrive/Masters_Project
!python scripts/run_experiments.py --exp exp_01 --limit 5 --no-4bit -v

## 4. Full Dataset C (required)

100 questions × systems. Slow on T4. Do **not** interrupt mid-run if possible; each experiment writes its own timestamped folder.

- `exp_01`: `baseline_llm` vs `rag_only`
- `exp_02`: `rag_only` vs `finetuned_rag` (needs `experiments/results/adapters/colab_t4_qlora_v1/adapter`)

How to read results: higher point coverage / token F1 / ROUGE-L is better *as a lexical proxy*. Latency is wall-clock `ask()` time. OOD items test refusal, not factual coverage.

In [ ]:
%cd /content/drive/MyDrive/Masters_Project
!python scripts/run_experiments.py --exp exp_01,exp_02 --no-4bit -v
!python scripts/build_results_tables.py

## 5. What the outputs mean

| Artefact | Meaning |
|----------|---------|
| `experiments/results/evaluation/exp_01_baseline_vs_rag_*/summary.json` | System A vs B automatic metrics |
| `experiments/results/evaluation/exp_02_rag_vs_finetuned_rag_*/summary.json` | System B vs C automatic metrics |
| `experiments/results/tables/results.md` | Thesis tables (placeholders remain until GPU runs exist) |
| `experiments/results/tables/figures/*.png` | Training + retrieval charts from **executed** data |

If GPU time runs out, document the limitation. Do not copy mock-LLM numbers into Tables 3–4.